In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Donation_Appeal_MASTER_FINAL_v8_7100.csv to Donation_Appeal_MASTER_FINAL_v8_7100.csv


In [ ]:
!pip install -q sentence-transformers scikit-learn pandas numpy joblib

In [ ]:
import os; os.kill(os.getpid(), 9)

In [ ]:
import json, numpy as np, pandas as pd, joblib
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.svm import SVC
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import classification_report, f1_score, accuracy_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

OUTPUT_DIR = Path("/content/models_final")
OUTPUT_DIR.mkdir(exist_ok=True)
RANDOM_SEED = 42
label_order = ["low", "medium", "high"]

# Embedder already loaded — skip re-downloading
# X_train, X_val, X_test, train, val, test, encoder, y_train, y_val_enc
# already exist from the previous cell — reuse them

# --- Try LogisticRegression instead of SVC ---
print("=== Testing LogisticRegression (more stable than SVC on this data) ===")
best_c, best_f1 = 0.1, 0.0
for C in [0.01, 0.05, 0.1, 0.5, 1.0]:
    clf = LogisticRegression(C=C, class_weight="balanced", max_iter=2000,
                             random_state=RANDOM_SEED, solver="lbfgs")
    clf.fit(X_train, y_train)
    preds = encoder.inverse_transform(clf.predict(X_val))
    f1  = f1_score(val["quality_label"], preds, labels=label_order, average="macro", zero_division=0)
    acc = accuracy_score(val["quality_label"], preds)
    print(f"  C={C}  F1={f1:.4f}  Acc={acc:.4f}")
    if f1 > best_f1:
        best_f1, best_c = f1, C

print(f"\nBest C={best_c} — retraining on train+val...")
X_all = np.vstack([X_train, X_val])
y_all = np.concatenate([y_train, y_val_enc])

classifier = LogisticRegression(C=best_c, class_weight="balanced", max_iter=2000,
                                random_state=RANDOM_SEED, solver="lbfgs")
classifier.fit(X_all, y_all)

regressor = Ridge(alpha=1.0)
regressor.fit(X_all, np.concatenate([
    train["overall_quality_score"], val["overall_quality_score"]
]))
print("Trained.")

# Sanity check — probabilities must DIFFER across inputs
test_texts = [
    "A family in Kaduwela lost their home in the floods. Three children have no shelter. Please donate now.",
    "help needed",
    "Nimal and his children need food and clean water after the Kelani river flooded their home in Hanwella last Tuesday. Your donation of Rs.2000 provides a full emergency food pack. Give now.",
    "families affected by disaster need support",
    "කඩුවෙල පවුලකට ගංවතුරෙන් නිවහන අහිමි විය. ළමයින් තිදෙනාට රැකවරණය නැත. කරුණාකර දැන් පරිත්‍යාග කරන්න.",
]
print("\n=== SANITY CHECK — probabilities must differ ===")
all_same = True
prev_proba = None
for t in test_texts:
    X_t   = np.array([embedder.encode(t)])
    pred  = encoder.inverse_transform(classifier.predict(X_t))[0]
    score = float(regressor.predict(X_t)[0])
    proba = classifier.predict_proba(X_t)[0]
    conf  = float(proba.max())
    if prev_proba is not None and not np.allclose(proba, prev_proba, atol=0.01):
        all_same = False
    prev_proba = proba
    print(f"  [{pred}] score={score:.2f} conf={conf:.2f} proba={np.round(proba,3)} | {t[:55]}")

if all_same:
    print("\n❌ STILL BROKEN — all probabilities identical. Do not save.")
else:
    print("\n✅ Model is responding to input — saving...")
    joblib.dump(embedder,   OUTPUT_DIR / "appeal_quality_vectorizer.joblib")
    joblib.dump(classifier, OUTPUT_DIR / "appeal_quality_classifier.joblib")
    joblib.dump(encoder,    OUTPUT_DIR / "appeal_quality_label_encoder.joblib")
    joblib.dump(regressor,  OUTPUT_DIR / "appeal_quality_regressor.joblib")
    print(f"Saved to {OUTPUT_DIR}/")

    y_pred = encoder.inverse_transform(classifier.predict(X_test)).tolist()
    f1  = f1_score(test["quality_label"], y_pred, labels=label_order, average="macro", zero_division=0)
    acc = accuracy_score(test["quality_label"], y_pred)
    print(f"\nTest — F1:{f1:.4f}  Acc:{acc:.4f}")
    print(classification_report(test["quality_label"], y_pred, labels=label_order, zero_division=0))
    print("\nDownload files now.")

=== Testing LogisticRegression (more stable than SVC on this data) ===
  C=0.01  F1=0.6800  Acc=0.6789
  C=0.05  F1=0.7177  Acc=0.7108
  C=0.1  F1=0.7282  Acc=0.7202
  C=0.5  F1=0.7518  Acc=0.7437
  C=1.0  F1=0.7591  Acc=0.7512

Best C=1.0 — retraining on train+val...
Trained.

=== SANITY CHECK — probabilities must differ ===
  [medium] score=3.75 conf=0.51 proba=[0.485 0.    0.515] | A family in Kaduwela lost their home in the floods. Thr
  [low] score=-1.11 conf=1.00 proba=[0. 1. 0.] | help needed
  [medium] score=3.02 conf=0.53 proba=[0.255 0.216 0.529] | Nimal and his children need food and clean water after 
  [high] score=7.98 conf=1.00 proba=[1. 0. 0.] | families affected by disaster need support
  [medium] score=2.90 conf=0.99 proba=[0.005 0.001 0.994] | කඩුවෙල පවුලකට ගංවතුරෙන් නිවහන අහිමි විය. ළමයින් තිදෙනාට

✅ Model is responding to input — saving...
Saved to /content/models_final/

Test — F1:0.7683  Acc:0.7606
              precision    recall  f1-score   support

         l

In [ ]:
# Fix regressor — must train on same X_all as classifier
from sklearn.linear_model import Ridge
import numpy as np

regressor_fixed = Ridge(alpha=1.0)
regressor_fixed.fit(X_all, np.concatenate([
    train["overall_quality_score"], val["overall_quality_score"]
]))

# Verify range on sanity texts
print("=== REGRESSOR RANGE CHECK ===")
for t in test_texts:
    X_t = np.array([embedder.encode(t)])
    raw = float(regressor_fixed.predict(X_t)[0])
    clipped = round(max(1.0, min(5.0, raw)), 2)
    print(f"  raw={raw:.2f} clipped={clipped} | {t[:55]}")

# Overwrite with fixed regressor
import joblib
from pathlib import Path
OUTPUT_DIR = Path("/content/models_final")
joblib.dump(regressor_fixed, OUTPUT_DIR / "appeal_quality_regressor.joblib")
print("\nFixed regressor saved. Now download all four files.")

=== REGRESSOR RANGE CHECK ===
  raw=3.75 clipped=3.75 | A family in Kaduwela lost their home in the floods. Thr
  raw=-1.11 clipped=1.0 | help needed
  raw=3.02 clipped=3.02 | Nimal and his children need food and clean water after 
  raw=7.98 clipped=5.0 | families affected by disaster need support
  raw=2.90 clipped=2.9 | කඩුවෙල පවුලකට ගංවතුරෙන් නිවහන අහිමි විය. ළමයින් තිදෙනාට

Fixed regressor saved. Now download all four files.


In [ ]:
from google.colab import files
import os

for f in sorted(os.listdir('/content/models_final')):
    size_mb = os.path.getsize(f'/content/models_final/{f}') / 1024 / 1024
    print(f"Downloading {f} ({size_mb:.1f} MB)")
    files.download(f'/content/models_final/{f}')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**New Model Foe better traing for regressor**

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Donation_Appeal_MASTER_FINAL_v8_7100.csv to Donation_Appeal_MASTER_FINAL_v8_7100.csv


In [ ]:
import pandas as pd
import numpy as np
from sklearn import __version__ as sk_version
import sys

print(f"Python: {sys.version}")
print(f"sklearn: {sk_version}")
print(f"numpy: {np.__version__}")
print(f"pandas: {pd.__version__}")

# Load
df = pd.read_csv("Donation_Appeal_MASTER_FINAL_v8_7100.csv")
df = df.dropna(subset=["appeal_text", "quality_label", "overall_quality_score"])
df["appeal_text"] = df["appeal_text"].astype(str).str.strip()
df = df[df["appeal_text"].str.len() > 10].reset_index(drop=True)

print(f"\nClean rows: {len(df)}")

# ── Feature engineering preview ──────────────────────────────────────────────
def extract_features(df):
    f = pd.DataFrame()
    f["text_len"]         = df["appeal_text"].str.len()
    f["word_count"]       = df["appeal_text"].str.split().str.len()
    f["avg_word_len"]     = f["text_len"] / (f["word_count"] + 1)
    f["sentence_count"]   = df["appeal_text"].str.count(r"[.!?।]") + 1
    f["avg_sent_len"]     = f["word_count"] / (f["sentence_count"] + 1)
    f["exclamation_count"]= df["appeal_text"].str.count(r"!")
    f["question_count"]   = df["appeal_text"].str.count(r"\?")
    f["digit_ratio"]      = df["appeal_text"].str.count(r"\d") / (f["text_len"] + 1)
    f["upper_ratio"]      = df["appeal_text"].str.count(r"[A-Z]") / (f["text_len"] + 1)
    f["is_sinhala"]       = (df["language"] == "Sinhala").astype(int)
    f["is_tamil"]         = (df["language"] == "Tamil").astype(int)

    # Urgency/CTA signal words (language-agnostic — currency and numbers)
    f["has_currency"]     = df["appeal_text"].str.contains(r"[$£€Rs\d]", regex=True).astype(int)
    f["has_donate_word"]  = df["appeal_text"].str.contains(
        r"\b(donat|contribut|help|support|give|fund|රුපියල්|දීමනා|நன்கொடை)\b",
        case=False, regex=True
    ).astype(int)

    return f

features = extract_features(df)
print("\nEngineered features preview:")
print(features.describe().round(2))

print("\nMean features by quality label:")
features["label"] = df["quality_label"]
print(features.groupby("label").mean().round(3))

Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
sklearn: 1.6.1
numpy: 2.1.3
pandas: 2.2.3

Clean rows: 7100


/tmp/ipykernel_4317/212153373.py:36: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  f["has_donate_word"]  = df["appeal_text"].str.contains(



Engineered features preview:
       text_len  word_count  avg_word_len  sentence_count  avg_sent_len  \
count   7100.00     7100.00       7100.00         7100.00       7100.00   
mean     660.74      116.51          6.10            8.67          9.46   
std      918.14      168.20          1.27            9.94          8.65   
min       37.00        4.00          3.57            1.00          0.67   
25%      140.00       21.00          5.30            3.00          4.80   
50%      278.50       41.00          5.68            5.00          8.28   
75%      799.25      146.00          6.33           10.00         12.91   
max    16942.00     3122.00         14.71          154.00        532.00   

       exclamation_count  question_count  digit_ratio  upper_ratio  \
count            7100.00         7100.00      7100.00      7100.00   
mean                0.55            0.04         0.00         0.03   
std                 1.51            0.32         0.01         0.04   
min           

In [ ]:
import pandas as pd
import numpy as np
import joblib
import re

from sklearn.pipeline         import Pipeline
from sklearn.preprocessing    import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble         import RandomForestClassifier, GradientBoostingRegressor
from sklearn.calibration      import CalibratedClassifierCV
from sklearn.model_selection  import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics          import (classification_report, confusion_matrix,
                                      mean_absolute_error, r2_score)
from scipy.sparse             import hstack, csr_matrix

# ── 1. Load & clean ──────────────────────────────────────────────────────────
df = pd.read_csv("Donation_Appeal_MASTER_FINAL_v8_7100.csv")
df = df.dropna(subset=["appeal_text", "quality_label", "overall_quality_score"])
df["appeal_text"] = df["appeal_text"].astype(str).str.strip()
df = df[df["appeal_text"].str.len() > 10].reset_index(drop=True)

# ── 2. Feature engineering ───────────────────────────────────────────────────
def extract_handcrafted(df):
    t = df["appeal_text"]
    f = pd.DataFrame()
    f["text_len"]          = t.str.len()
    f["word_count"]        = t.str.split().str.len()
    f["avg_word_len"]      = f["text_len"] / (f["word_count"] + 1)
    f["sentence_count"]    = t.str.count(r"[.!?।]") + 1
    f["avg_sent_len"]      = f["word_count"] / (f["sentence_count"] + 1)
    f["exclamation_count"] = t.str.count(r"!")
    f["question_count"]    = t.str.count(r"\?")
    f["digit_ratio"]       = t.str.count(r"\d") / (f["text_len"] + 1)
    f["upper_ratio"]       = t.str.count(r"[A-Z]") / (f["text_len"] + 1)
    f["is_sinhala"]        = (df["language"] == "Sinhala").astype(int)
    f["is_tamil"]          = (df["language"] == "Tamil").astype(int)
    f["has_currency"]      = t.str.contains(r"[$£€Rs\d]", regex=True).astype(int)
    # Fix the regex warning — use non-capturing group
    f["has_donate_word"]   = t.str.contains(
        r"donat|contribut|help|support|give|fund|රුපියල්|දීමනා|நன்கொடை",
        case=False, regex=True
    ).astype(int)
    # Additional signals revealed by Step 1 audit
    f["log_text_len"]      = np.log1p(f["text_len"])    # compress the long-tail
    f["log_word_count"]    = np.log1p(f["word_count"])
    return f.fillna(0).values

X_hand = extract_handcrafted(df)

# ── 3. TF-IDF (character n-grams — language-agnostic, works for all 3 scripts)
#    Word n-grams fail on Sinhala/Tamil because tokenisation differs.
#    Char n-grams capture subword patterns across all three languages.
tfidf = TfidfVectorizer(
    analyzer    = "char_wb",   # character n-grams within word boundaries
    ngram_range = (3, 5),
    max_features= 30_000,
    sublinear_tf= True,        # log(tf) — reduces dominance of long English texts
    strip_accents= None,       # preserve Sinhala/Tamil characters
    min_df      = 3,
)
X_tfidf = tfidf.fit_transform(df["appeal_text"])

# ── 4. Combine: sparse TF-IDF + dense handcrafted ───────────────────────────
X_hand_sparse = csr_matrix(StandardScaler().fit_transform(X_hand))
X_combined    = hstack([X_tfidf, X_hand_sparse])

# ── 5. Labels & scores ───────────────────────────────────────────────────────
le = LabelEncoder()
y  = le.fit_transform(df["quality_label"])   # low=0, medium=1, high=2 (alphabetical)
s  = df["overall_quality_score"].values.astype(float)

print("Label encoding:", dict(zip(le.classes_, le.transform(le.classes_))))

# ── 6. Train / test split — stratified ──────────────────────────────────────
X_tr, X_te, y_tr, y_te, s_tr, s_te = train_test_split(
    X_combined, y, s,
    test_size   = 0.2,
    random_state= 42,
    stratify    = y,
)
print(f"Train: {X_tr.shape[0]}  |  Test: {X_te.shape[0]}")

# ── 7. Classifier — Random Forest + isotonic calibration ────────────────────
#    RF is chosen over GBM here because:
#    - Naturally produces reasonable probabilities before calibration
#    - Faster to train on sparse matrices
#    - Isotonic calibration on top fixes the confidence problem directly
base_clf = RandomForestClassifier(
    n_estimators     = 500,
    max_depth        = None,
    min_samples_leaf = 3,
    max_features     = "sqrt",
    class_weight     = "balanced",  # handles any residual imbalance
    random_state     = 42,
    n_jobs           = -1,
)

# cv=5 means calibration is fit on held-out folds — prevents overconfidence
clf = CalibratedClassifierCV(base_clf, method="isotonic", cv=5)
clf.fit(X_tr, y_tr)

y_pred     = clf.predict(X_te)
y_proba    = clf.predict_proba(X_te)
confidence = y_proba.max(axis=1)

print("\n── Classifier Results ──────────────────────────────────────────")
print(classification_report(y_te, y_pred, target_names=le.classes_))
print("Confusion matrix (rows=actual, cols=predicted):")
print(pd.DataFrame(
    confusion_matrix(y_te, y_pred),
    index   = [f"actual_{c}"    for c in le.classes_],
    columns = [f"pred_{c}" for c in le.classes_],
))
print(f"\nMean confidence (all):     {confidence.mean():.3f}")
print(f"Mean confidence (correct): {confidence[y_pred == y_te].mean():.3f}")
print(f"Mean confidence (wrong):   {confidence[y_pred != y_te].mean():.3f}")

# ── 8. 5-fold CV accuracy ────────────────────────────────────────────────────
# Use base_clf (uncalibrated) for CV speed — calibration wrapper is slow in CV
cv_scores = cross_val_score(base_clf, X_combined, y, cv=5, scoring="accuracy", n_jobs=-1)
print(f"\n5-fold CV accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
print(f"Per-fold:           {np.round(cv_scores, 3)}")

# ── 9. Per-language breakdown ────────────────────────────────────────────────
# Re-split test indices to evaluate per language
_, te_idx = train_test_split(
    np.arange(len(df)), test_size=0.2, random_state=42, stratify=y
)
te_df = df.iloc[te_idx].copy()
te_df["pred_label"] = le.inverse_transform(y_pred)
te_df["correct"]    = (y_pred == y_te)

print("\n── Per-language accuracy ───────────────────────────────────────")
print(te_df.groupby("language")["correct"].agg(["mean", "sum", "count"]).round(3))

# ── 10. Regressor — trained on classifier probabilities + features ───────────
proba_tr = clf.predict_proba(X_tr)
proba_te = clf.predict_proba(X_te)

# Dense handcrafted features for regressor (already scaled above — redo cleanly)
scaler   = StandardScaler()
hand_tr  = scaler.fit_transform(extract_handcrafted(df.iloc[
    train_test_split(np.arange(len(df)), test_size=0.2, random_state=42, stratify=y)[0]
]))
hand_te  = scaler.transform(extract_handcrafted(df.iloc[te_idx]))

X_reg_tr = np.hstack([proba_tr, hand_tr])
X_reg_te = np.hstack([proba_te, hand_te])

reg = GradientBoostingRegressor(
    n_estimators  = 300,
    max_depth     = 4,
    learning_rate = 0.05,
    subsample     = 0.8,
    random_state  = 42,
)
reg.fit(X_reg_tr, s_tr)

s_pred = reg.predict(X_reg_te)
s_pred = np.clip(s_pred, 1.0, 5.0)

print("\n── Regressor Results ───────────────────────────────────────────")
print(f"MAE : {mean_absolute_error(s_te, s_pred):.3f}")
print(f"R²  : {r2_score(s_te, s_pred):.3f}")

# Score vs predicted rounded agreement
rounded_match = (np.round(s_pred) == s_te).mean()
print(f"Rounded score match: {rounded_match:.3f}")

# ── 11. Classifier / regressor agreement check ───────────────────────────────
# Check: does the score's implied label match the classifier label?
def score_to_label(score):
    if score <= 2: return "low"
    if score <= 3: return "medium"
    return "high"

implied   = np.array([score_to_label(s) for s in s_pred])
clf_label = le.inverse_transform(y_pred)
agreement = (implied == clf_label).mean()
print(f"Classifier / regressor label agreement: {agreement:.3f}")

Label encoding: {'high': np.int64(0), 'low': np.int64(1), 'medium': np.int64(2)}
Train: 5680  |  Test: 1420

── Classifier Results ──────────────────────────────────────────
              precision    recall  f1-score   support

        high       0.77      0.87      0.82       492
         low       0.95      0.93      0.94       420
      medium       0.80      0.71      0.76       508

    accuracy                           0.83      1420
   macro avg       0.84      0.84      0.84      1420
weighted avg       0.84      0.83      0.83      1420

Confusion matrix (rows=actual, cols=predicted):
               pred_high  pred_low  pred_medium
actual_high          428         1           63
actual_low             1       392           27
actual_medium        127        18          363

Mean confidence (all):     0.843
Mean confidence (correct): 0.880
Mean confidence (wrong):   0.660

5-fold CV accuracy: 0.771 ± 0.121
Per-fold:           [0.676 0.587 0.911 0.817 0.864]

── Per-language a

In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing         import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble              import RandomForestClassifier, GradientBoostingRegressor
from sklearn.calibration           import CalibratedClassifierCV
from sklearn.model_selection       import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics               import (classification_report, confusion_matrix,
                                           mean_absolute_error, r2_score)
from scipy.sparse                  import hstack, csr_matrix

# ── 1. Load ──────────────────────────────────────────────────────────────────
df = pd.read_csv("Donation_Appeal_MASTER_FINAL_v8_7100.csv")
df = df.dropna(subset=["appeal_text", "quality_label", "overall_quality_score"])
df["appeal_text"] = df["appeal_text"].astype(str).str.strip()
df = df[df["appeal_text"].str.len() > 10].reset_index(drop=True)

# ── 2. Downsample Sinhala/Tamil augmented rows ───────────────────────────────
# The augmented rows are too internally consistent — the model memorises their
# patterns rather than learning quality signal. We keep all human-reviewed rows
# and reduce the augmented ones so they don't dominate Sinhala/Tamil.
AUGMENTED_ORIGINS = {
    "controlled_augmentation_native_reviewed",
    "controlled_augmentation_from_original_english",
    "AI_GENERATED_BOUNDARY_CASE",
    "AI_GENERATED_NOISY_CASE",
}

non_english_augmented = df[
    (df["language"].isin(["Sinhala", "Tamil"])) &
    (df["data_origin"].isin(AUGMENTED_ORIGINS))
]
rest = df[~df.index.isin(non_english_augmented.index)]

# Keep 60% of augmented non-English rows — enough signal, less memorisation
aug_sample = non_english_augmented.groupby(
    ["language", "quality_label"], group_keys=False
).apply(lambda g: g.sample(frac=0.60, random_state=42))

df_clean = pd.concat([rest, aug_sample]).reset_index(drop=True)

print(f"Original rows : {len(df)}")
print(f"After sampling: {len(df_clean)}")
print("\nLanguage distribution after sampling:")
print(df_clean["language"].value_counts())
print("\nLabel distribution after sampling:")
print(df_clean["quality_label"].value_counts())

# ── 3. Feature engineering ───────────────────────────────────────────────────
def extract_handcrafted(frame):
    t = frame["appeal_text"]
    f = pd.DataFrame(index=frame.index)
    f["log_text_len"]      = np.log1p(t.str.len())
    f["log_word_count"]    = np.log1p(t.str.split().str.len())
    f["avg_word_len"]      = t.str.len() / (t.str.split().str.len() + 1)
    f["sentence_count"]    = np.log1p(t.str.count(r"[.!?।]") + 1)
    f["avg_sent_len"]      = t.str.split().str.len() / (t.str.count(r"[.!?।]") + 2)
    f["exclamation_count"] = np.log1p(t.str.count(r"!"))
    f["question_count"]    = np.log1p(t.str.count(r"\?"))
    f["digit_ratio"]       = t.str.count(r"\d") / (t.str.len() + 1)
    f["upper_ratio"]       = t.str.count(r"[A-Z]") / (t.str.len() + 1)
    f["has_currency"]      = t.str.contains(r"[$£€Rs\d]", regex=True).astype(int)
    f["has_donate_word"]   = t.str.contains(
        r"donat|contribut|help|support|give|fund|රුපියල්|දීමනා|நன்கொடை",
        case=False, regex=True
    ).astype(int)
    # Language flags — kept so classifier knows script context
    f["is_sinhala"]        = (frame["language"] == "Sinhala").astype(int)
    f["is_tamil"]          = (frame["language"] == "Tamil").astype(int)
    return f.fillna(0).values

# ── 4. Separate TF-IDF per language group ────────────────────────────────────
# This is the key fix for the variance/accuracy inversion problem.
# A single TF-IDF across all three scripts learns script identity as a proxy
# for quality — Sinhala/Tamil become trivially separable because their
# augmented patterns are uniform. Separate vectorisers force each to learn
# quality signal within its own script.

def build_tfidf(texts, analyzer, ngram_range, max_features):
    vec = TfidfVectorizer(
        analyzer     = analyzer,
        ngram_range  = ngram_range,
        max_features = max_features,
        sublinear_tf = True,
        strip_accents= None,
        min_df       = 3,
    )
    return vec, vec.fit_transform(texts)

en_mask = df_clean["language"] == "English"
si_mask = df_clean["language"] == "Sinhala"
ta_mask = df_clean["language"] == "Tamil"

# English: word n-grams work well, larger vocab
tfidf_en, X_en = build_tfidf(df_clean.loc[en_mask, "appeal_text"], "word",   (1, 2), 25_000)
# Sinhala: char n-grams, smaller vocab
tfidf_si, X_si = build_tfidf(df_clean.loc[si_mask, "appeal_text"], "char_wb",(3, 5),  8_000)
# Tamil: char n-grams, smaller vocab
tfidf_ta, X_ta = build_tfidf(df_clean.loc[ta_mask, "appeal_text"], "char_wb",(3, 5),  8_000)

print(f"\nTF-IDF shapes — EN: {X_en.shape}, SI: {X_si.shape}, TA: {X_ta.shape}")

# ── 5. Assemble combined feature matrix (row order preserved) ─────────────────
# Stack handcrafted features, then slot TF-IDF features into language-specific
# columns. Non-applicable language columns are zero-filled.
scaler    = StandardScaler()
X_hand    = scaler.fit_transform(extract_handcrafted(df_clean))
X_hand_sp = csr_matrix(X_hand)

# Build per-language sparse blocks aligned to full dataset row order
from scipy.sparse import lil_matrix

def expand_to_full(sparse_block, mask, n_rows):
    """Insert sparse_block rows back into their original positions."""
    full = lil_matrix((n_rows, sparse_block.shape[1]))
    idx  = np.where(mask)[0]
    for i, row_i in enumerate(idx):
        full[row_i] = sparse_block[i]
    return full.tocsr()

n = len(df_clean)
X_en_full = expand_to_full(X_en, en_mask.values, n)
X_si_full = expand_to_full(X_si, si_mask.values, n)
X_ta_full = expand_to_full(X_ta, ta_mask.values, n)

X_combined = hstack([X_hand_sp, X_en_full, X_si_full, X_ta_full])
print(f"Combined feature matrix: {X_combined.shape}")

# ── 6. Labels ────────────────────────────────────────────────────────────────
le = LabelEncoder()
y  = le.fit_transform(df_clean["quality_label"])
s  = df_clean["overall_quality_score"].values.astype(float)
print("\nLabel encoding:", dict(zip(le.classes_, le.transform(le.classes_))))

# ── 7. Train/test split ───────────────────────────────────────────────────────
X_tr, X_te, y_tr, y_te, s_tr, s_te, idx_tr, idx_te = train_test_split(
    X_combined, y, s, np.arange(n),
    test_size    = 0.2,
    random_state = 42,
    stratify     = y,
)
print(f"Train: {X_tr.shape[0]}  |  Test: {X_te.shape[0]}")

# ── 8. Classifier ─────────────────────────────────────────────────────────────
base_clf = RandomForestClassifier(
    n_estimators     = 500,
    max_depth        = 20,          # cap depth — reduces overfitting to augmented patterns
    min_samples_leaf = 5,           # was 3 — forces more generalisation
    max_features     = "sqrt",
    class_weight     = "balanced",
    random_state     = 42,
    n_jobs           = -1,
)
clf = CalibratedClassifierCV(base_clf, method="isotonic", cv=5)
clf.fit(X_tr, y_tr)

y_pred  = clf.predict(X_te)
y_proba = clf.predict_proba(X_te)
conf    = y_proba.max(axis=1)

print("\n── Classifier Results ──────────────────────────────────────────")
print(classification_report(y_te, y_pred, target_names=le.classes_))
print("Confusion matrix:")
print(pd.DataFrame(
    confusion_matrix(y_te, y_pred),
    index   = [f"actual_{c}" for c in le.classes_],
    columns = [f"pred_{c}"   for c in le.classes_],
))
print(f"\nMean confidence (all):     {conf.mean():.3f}")
print(f"Mean confidence (correct): {conf[y_pred == y_te].mean():.3f}")
print(f"Mean confidence (wrong):   {conf[y_pred != y_te].mean():.3f}")

# ── 9. CV on base classifier ──────────────────────────────────────────────────
cv   = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cvs  = cross_val_score(base_clf, X_combined, y, cv=cv, scoring="accuracy", n_jobs=-1)
print(f"\n5-fold CV accuracy: {cvs.mean():.3f} ± {cvs.std():.3f}")
print(f"Per-fold:           {np.round(cvs, 3)}")

# ── 10. Per-language accuracy ─────────────────────────────────────────────────
te_df = df_clean.iloc[idx_te].copy()
te_df["pred_label"] = le.inverse_transform(y_pred)
te_df["correct"]    = (y_pred == y_te)
print("\n── Per-language accuracy ───────────────────────────────────────")
print(te_df.groupby("language")["correct"].agg(["mean", "sum", "count"]).round(3))

# ── 11. Regressor ─────────────────────────────────────────────────────────────
# Joint input: calibrated class probabilities + handcrafted features (dense only)
# We use dense handcrafted features for the regressor — TF-IDF already informs
# the classifier whose probabilities the regressor receives.
hand_tr = scaler.transform(extract_handcrafted(df_clean.iloc[idx_tr]))
hand_te = scaler.transform(extract_handcrafted(df_clean.iloc[idx_te]))

X_reg_tr = np.hstack([clf.predict_proba(X_tr), hand_tr])
X_reg_te = np.hstack([clf.predict_proba(X_te), hand_te])

reg = GradientBoostingRegressor(
    n_estimators  = 300,
    max_depth     = 4,
    learning_rate = 0.05,
    subsample     = 0.8,
    random_state  = 42,
)
reg.fit(X_reg_tr, s_tr)
s_pred = np.clip(reg.predict(X_reg_te), 1.0, 5.0)

print("\n── Regressor Results ───────────────────────────────────────────")
print(f"MAE : {mean_absolute_error(s_te, s_pred):.3f}")
print(f"R²  : {r2_score(s_te, s_pred):.3f}")
print(f"Rounded score match: {(np.round(s_pred) == s_te).mean():.3f}")

def score_to_label(score):
    if score <= 2: return "low"
    if score <= 3: return "medium"
    return "high"

implied   = np.array([score_to_label(s) for s in s_pred])
clf_label = le.inverse_transform(y_pred)
agreement = (implied == clf_label).mean()
print(f"Classifier / regressor label agreement: {agreement:.3f}")

# ── 12. Per-language regressor breakdown ──────────────────────────────────────
te_df["pred_score"] = s_pred
te_df["true_score"] = s_te
print("\n── Per-language regressor MAE ──────────────────────────────────")
for lang, grp in te_df.groupby("language"):
    mae = mean_absolute_error(grp["true_score"], grp["pred_score"])
    print(f"  {lang:8s}: MAE={mae:.3f}  n={len(grp)}")

/tmp/ipykernel_4317/2870010224.py:40: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(lambda g: g.sample(frac=0.60, random_state=42))


Original rows : 7100
After sampling: 6121

Language distribution after sampling:
language
English    4600
Tamil       762
Sinhala     759
Name: count, dtype: int64

Label distribution after sampling:
quality_label
medium    2166
high      2133
low       1822
Name: count, dtype: int64

TF-IDF shapes — EN: (4600, 25000), SI: (759, 7571), TA: (762, 7310)
Combined feature matrix: (6121, 39894)

Label encoding: {'high': np.int64(0), 'low': np.int64(1), 'medium': np.int64(2)}
Train: 4896  |  Test: 1225

── Classifier Results ──────────────────────────────────────────
              precision    recall  f1-score   support

        high       0.78      0.76      0.77       427
         low       0.92      0.95      0.93       365
      medium       0.74      0.73      0.73       433

    accuracy                           0.81      1225
   macro avg       0.81      0.81      0.81      1225
weighted avg       0.80      0.81      0.80      1225

Confusion matrix:
               pred_high  pred_lo

In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing           import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble                import RandomForestClassifier, GradientBoostingRegressor
from sklearn.calibration             import CalibratedClassifierCV
from sklearn.model_selection         import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics                 import (classification_report, confusion_matrix,
                                             mean_absolute_error, r2_score)
from sklearn.decomposition           import TruncatedSVD
from scipy.sparse                    import hstack, csr_matrix, lil_matrix

# ── 1. Load ───────────────────────────────────────────────────────────────────
df = pd.read_csv("Donation_Appeal_MASTER_FINAL_v8_7100.csv")
df = df.dropna(subset=["appeal_text", "quality_label", "overall_quality_score"])
df["appeal_text"] = df["appeal_text"].astype(str).str.strip()
df = df[df["appeal_text"].str.len() > 10].reset_index(drop=True)

# ── 2. Downsample augmented non-English to 40% ────────────────────────────────
AUGMENTED_ORIGINS = {
    "controlled_augmentation_native_reviewed",
    "controlled_augmentation_from_original_english",
    "AI_GENERATED_BOUNDARY_CASE",
    "AI_GENERATED_NOISY_CASE",
}
non_english_aug = df[
    (df["language"].isin(["Sinhala", "Tamil"])) &
    (df["data_origin"].isin(AUGMENTED_ORIGINS))
]
rest        = df[~df.index.isin(non_english_aug.index)]
aug_sample  = non_english_aug.groupby(
    ["language", "quality_label"], group_keys=False
).apply(lambda g: g.sample(frac=0.40, random_state=42), include_groups=False)

# include_groups=False fixes the DeprecationWarning from Step 3
df_clean = pd.concat([rest, df.loc[aug_sample.index]]).reset_index(drop=True)

print(f"Rows after sampling: {len(df_clean)}")
print(df_clean["language"].value_counts())
print(df_clean["quality_label"].value_counts())

# ── 3. Handcrafted features ───────────────────────────────────────────────────
def extract_handcrafted(frame):
    t = frame["appeal_text"].astype(str)
    f = pd.DataFrame(index=frame.index)
    f["log_text_len"]      = np.log1p(t.str.len())
    f["log_word_count"]    = np.log1p(t.str.split().str.len())
    f["avg_word_len"]      = t.str.len() / (t.str.split().str.len() + 1)
    f["sentence_count"]    = np.log1p(t.str.count(r"[.!?।]") + 1)
    f["avg_sent_len"]      = t.str.split().str.len() / (t.str.count(r"[.!?།]") + 2)
    f["exclamation_count"] = np.log1p(t.str.count(r"!"))
    f["question_count"]    = np.log1p(t.str.count(r"\?"))
    f["digit_ratio"]       = t.str.count(r"\d") / (t.str.len() + 1)
    f["upper_ratio"]       = t.str.count(r"[A-Z]") / (t.str.len() + 1)
    f["has_currency"]      = t.str.contains(r"[$£€Rs\d]", regex=True).astype(int)
    f["has_donate_word"]   = t.str.contains(
        r"donat|contribut|help|support|give|fund|රුපියල්|දීමනා|நன்கொடை",
        case=False, regex=True
    ).astype(int)
    f["is_sinhala"]        = (frame["language"] == "Sinhala").astype(int)
    f["is_tamil"]          = (frame["language"] == "Tamil").astype(int)
    return f.fillna(0).values

# ── 4. Per-language TF-IDF ────────────────────────────────────────────────────
en_mask = df_clean["language"] == "English"
si_mask = df_clean["language"] == "Sinhala"
ta_mask = df_clean["language"] == "Tamil"

def fit_tfidf(texts, analyzer, ngram_range, max_features):
    vec = TfidfVectorizer(
        analyzer=analyzer, ngram_range=ngram_range,
        max_features=max_features, sublinear_tf=True,
        strip_accents=None, min_df=3,
    )
    return vec, vec.fit_transform(texts)

tfidf_en, X_en = fit_tfidf(df_clean.loc[en_mask, "appeal_text"], "word",    (1, 2), 25_000)
tfidf_si, X_si = fit_tfidf(df_clean.loc[si_mask, "appeal_text"], "char_wb", (3, 5),  8_000)
tfidf_ta, X_ta = fit_tfidf(df_clean.loc[ta_mask, "appeal_text"], "char_wb", (3, 5),  8_000)

# ── 5. Reduce TF-IDF dims for regressor input (LSA) ──────────────────────────
# The regressor works on dense input. We compress each TF-IDF block to
# 50 LSA components — enough to carry quality signal without the sparsity
# that kills gradient boosting performance.
N_COMPONENTS = 50

svd_en = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)
svd_si = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)
svd_ta = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)

lsa_en = svd_en.fit_transform(X_en)   # (n_en, 50)
lsa_si = svd_si.fit_transform(X_si)   # (n_si, 50)
lsa_ta = svd_ta.fit_transform(X_ta)   # (n_ta, 50)

print(f"\nLSA variance explained — EN: {svd_en.explained_variance_ratio_.sum():.3f}, "
      f"SI: {svd_si.explained_variance_ratio_.sum():.3f}, "
      f"TA: {svd_ta.explained_variance_ratio_.sum():.3f}")

# ── 6. Expand sparse TF-IDF to full matrix for classifier ─────────────────────
def expand_sparse(block, mask_values, n_rows):
    full = lil_matrix((n_rows, block.shape[1]))
    for i, row_i in enumerate(np.where(mask_values)[0]):
        full[row_i] = block[i]
    return full.tocsr()

n = len(df_clean)
X_en_full = expand_sparse(X_en, en_mask.values, n)
X_si_full = expand_sparse(X_si, si_mask.values, n)
X_ta_full = expand_sparse(X_ta, ta_mask.values, n)

# Expand LSA blocks to full dense matrix for regressor
def expand_dense(block, mask_values, n_rows, n_cols):
    full = np.zeros((n_rows, n_cols))
    full[np.where(mask_values)[0]] = block
    return full

lsa_en_full = expand_dense(lsa_en, en_mask.values, n, N_COMPONENTS)
lsa_si_full = expand_dense(lsa_si, si_mask.values, n, N_COMPONENTS)
lsa_ta_full = expand_dense(lsa_ta, ta_mask.values, n, N_COMPONENTS)

scaler  = StandardScaler()
X_hand  = scaler.fit_transform(extract_handcrafted(df_clean))

# Classifier input: sparse TF-IDF blocks + handcrafted
X_clf = hstack([csr_matrix(X_hand), X_en_full, X_si_full, X_ta_full])

# Regressor base features: LSA + handcrafted (dense, no class proba yet)
X_reg_base = np.hstack([X_hand, lsa_en_full, lsa_si_full, lsa_ta_full])

print(f"Classifier input shape : {X_clf.shape}")
print(f"Regressor base shape   : {X_reg_base.shape}")

# ── 7. Labels ─────────────────────────────────────────────────────────────────
le = LabelEncoder()
y  = le.fit_transform(df_clean["quality_label"])
s  = df_clean["overall_quality_score"].values.astype(float)
print("\nLabel encoding:", dict(zip(le.classes_, le.transform(le.classes_))))

# ── 8. Train/test split ───────────────────────────────────────────────────────
idx = np.arange(n)
(X_clf_tr, X_clf_te,
 X_reg_tr,  X_reg_te,
 y_tr, y_te, s_tr, s_te,
 idx_tr, idx_te) = train_test_split(
    X_clf, X_reg_base, y, s, idx,
    test_size=0.2, random_state=42, stratify=y,
)
print(f"Train: {X_clf_tr.shape[0]}  |  Test: {X_clf_te.shape[0]}")

# ── 9. Classifier ─────────────────────────────────────────────────────────────
base_clf = RandomForestClassifier(
    n_estimators     = 500,
    max_depth        = 20,
    min_samples_leaf = 5,
    max_features     = "sqrt",
    class_weight     = "balanced",
    random_state     = 42,
    n_jobs           = -1,
)
clf = CalibratedClassifierCV(base_clf, method="isotonic", cv=5)
clf.fit(X_clf_tr, y_tr)

y_pred  = clf.predict(X_clf_te)
y_proba = clf.predict_proba(X_clf_te)
conf    = y_proba.max(axis=1)

print("\n── Classifier Results ───────────────────────────────────────────")
print(classification_report(y_te, y_pred, target_names=le.classes_))
print("Confusion matrix:")
print(pd.DataFrame(
    confusion_matrix(y_te, y_pred),
    index   = [f"actual_{c}" for c in le.classes_],
    columns = [f"pred_{c}"   for c in le.classes_],
))
print(f"\nMean confidence (all):     {conf.mean():.3f}")
print(f"Mean confidence (correct): {conf[y_pred == y_te].mean():.3f}")
print(f"Mean confidence (wrong):   {conf[y_pred != y_te].mean():.3f}")

# ── 10. CV ────────────────────────────────────────────────────────────────────
cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cvs = cross_val_score(base_clf, X_clf, y, cv=cv, scoring="accuracy", n_jobs=-1)
print(f"\n5-fold CV accuracy: {cvs.mean():.3f} ± {cvs.std():.3f}")
print(f"Per-fold:           {np.round(cvs, 3)}")

# ── 11. Per-language accuracy ─────────────────────────────────────────────────
te_df = df_clean.iloc[idx_te].copy()
te_df["pred_label"] = le.inverse_transform(y_pred)
te_df["correct"]    = (y_pred == y_te)
print("\n── Per-language accuracy ────────────────────────────────────────")
print(te_df.groupby("language")["correct"].agg(["mean", "sum", "count"]).round(3))

# ── 12. Regressor — now informed by LSA + class probabilities ─────────────────
proba_tr = clf.predict_proba(X_clf_tr)   # (n_train, 3)
proba_te = clf.predict_proba(X_clf_te)   # (n_test,  3)

X_full_reg_tr = np.hstack([proba_tr, X_reg_tr])
X_full_reg_te = np.hstack([proba_te, X_reg_te])

reg = GradientBoostingRegressor(
    n_estimators  = 300,
    max_depth     = 4,
    learning_rate = 0.05,
    subsample     = 0.8,
    random_state  = 42,
)
reg.fit(X_full_reg_tr, s_tr)
s_pred = np.clip(reg.predict(X_full_reg_te), 1.0, 5.0)

print("\n── Regressor Results ────────────────────────────────────────────")
print(f"MAE : {mean_absolute_error(s_te, s_pred):.3f}")
print(f"R²  : {r2_score(s_te, s_pred):.3f}")
print(f"Rounded score match: {(np.round(s_pred) == s_te).mean():.3f}")

def score_to_label(score):
    if score <= 2: return "low"
    if score <= 3: return "medium"
    return "high"

implied   = np.array([score_to_label(v) for v in s_pred])
clf_label = le.inverse_transform(y_pred)
agreement = (implied == clf_label).mean()
print(f"Classifier / regressor label agreement: {agreement:.3f}")

print("\n── Per-language regressor MAE ───────────────────────────────────")
te_df["pred_score"] = s_pred
te_df["true_score"] = s_te
for lang, grp in te_df.groupby("language"):
    mae = mean_absolute_error(grp["true_score"], grp["pred_score"])
    print(f"  {lang:8s}: MAE={mae:.3f}  n={len(grp)}")

Rows after sampling: 5627
language
English    4600
Tamil       516
Sinhala     511
Name: count, dtype: int64
quality_label
medium    1979
high      1968
low       1680
Name: count, dtype: int64

LSA variance explained — EN: 0.105, SI: 0.561, TA: 0.569
Classifier input shape : (5627, 39225)
Regressor base shape   : (5627, 163)

Label encoding: {'high': np.int64(0), 'low': np.int64(1), 'medium': np.int64(2)}
Train: 4501  |  Test: 1126

── Classifier Results ───────────────────────────────────────────
              precision    recall  f1-score   support

        high       0.80      0.75      0.78       394
         low       0.90      0.93      0.91       336
      medium       0.71      0.74      0.73       396

    accuracy                           0.80      1126
   macro avg       0.81      0.81      0.81      1126
weighted avg       0.80      0.80      0.80      1126

Confusion matrix:
               pred_high  pred_low  pred_medium
actual_high          296         1           97
a

In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing           import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble                import RandomForestClassifier, GradientBoostingRegressor
from sklearn.calibration             import CalibratedClassifierCV
from sklearn.isotonic                import IsotonicRegression
from sklearn.model_selection         import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics                 import (classification_report, confusion_matrix,
                                             mean_absolute_error, r2_score)
from sklearn.decomposition           import TruncatedSVD
from scipy.sparse                    import hstack, csr_matrix, lil_matrix

# ── 1. Load ───────────────────────────────────────────────────────────────────
df = pd.read_csv("Donation_Appeal_MASTER_FINAL_v8_7100.csv")
df = df.dropna(subset=["appeal_text", "quality_label", "overall_quality_score"])
df["appeal_text"] = df["appeal_text"].astype(str).str.strip()
df = df[df["appeal_text"].str.len() > 10].reset_index(drop=True)

# ── 2. Downsample augmented non-English (40%) ─────────────────────────────────
AUGMENTED_ORIGINS = {
    "controlled_augmentation_native_reviewed",
    "controlled_augmentation_from_original_english",
    "AI_GENERATED_BOUNDARY_CASE",
    "AI_GENERATED_NOISY_CASE",
}
non_english_aug = df[
    (df["language"].isin(["Sinhala", "Tamil"])) &
    (df["data_origin"].isin(AUGMENTED_ORIGINS))
]
rest       = df[~df.index.isin(non_english_aug.index)]
aug_sample = non_english_aug.groupby(
    ["language", "quality_label"], group_keys=False
).apply(lambda g: g.sample(frac=0.40, random_state=42), include_groups=False)
df_clean = pd.concat([rest, df.loc[aug_sample.index]]).reset_index(drop=True)

print(f"Rows: {len(df_clean)}")
print(df_clean["language"].value_counts())

# ── 3. Handcrafted features ───────────────────────────────────────────────────
def extract_handcrafted(frame):
    t = frame["appeal_text"].astype(str)
    f = pd.DataFrame(index=frame.index)
    f["log_text_len"]      = np.log1p(t.str.len())
    f["log_word_count"]    = np.log1p(t.str.split().str.len())
    f["avg_word_len"]      = t.str.len() / (t.str.split().str.len() + 1)
    f["sentence_count"]    = np.log1p(t.str.count(r"[.!?।]") + 1)
    f["avg_sent_len"]      = t.str.split().str.len() / (t.str.count(r"[.!?།]") + 2)
    f["exclamation_count"] = np.log1p(t.str.count(r"!"))
    f["question_count"]    = np.log1p(t.str.count(r"\?"))
    f["digit_ratio"]       = t.str.count(r"\d") / (t.str.len() + 1)
    f["upper_ratio"]       = t.str.count(r"[A-Z]") / (t.str.len() + 1)
    f["has_currency"]      = t.str.contains(r"[$£€Rs\d]", regex=True).astype(int)
    f["has_donate_word"]   = t.str.contains(
        r"donat|contribut|help|support|give|fund|රුපියල්|දීමනා|நன்கொடை",
        case=False, regex=True
    ).astype(int)
    f["is_sinhala"]        = (frame["language"] == "Sinhala").astype(int)
    f["is_tamil"]          = (frame["language"] == "Tamil").astype(int)
    return f.fillna(0).values

# ── 4. Per-language TF-IDF + LSA ─────────────────────────────────────────────
en_mask = df_clean["language"] == "English"
si_mask = df_clean["language"] == "Sinhala"
ta_mask = df_clean["language"] == "Tamil"

def fit_tfidf(texts, analyzer, ngram_range, max_features):
    vec = TfidfVectorizer(
        analyzer=analyzer, ngram_range=ngram_range,
        max_features=max_features, sublinear_tf=True,
        strip_accents=None, min_df=3,
    )
    return vec, vec.fit_transform(texts)

tfidf_en, X_en = fit_tfidf(df_clean.loc[en_mask, "appeal_text"], "word",    (1, 2), 25_000)
tfidf_si, X_si = fit_tfidf(df_clean.loc[si_mask, "appeal_text"], "char_wb", (3, 5),  8_000)
tfidf_ta, X_ta = fit_tfidf(df_clean.loc[ta_mask, "appeal_text"], "char_wb", (3, 5),  8_000)

# English gets 150 LSA components — its 25k vocab needs more dimensions
# Sinhala/Tamil keep 50 — their smaller vocab is well covered
svd_en = TruncatedSVD(n_components=150, random_state=42)
svd_si = TruncatedSVD(n_components=50,  random_state=42)
svd_ta = TruncatedSVD(n_components=50,  random_state=42)

lsa_en = svd_en.fit_transform(X_en)
lsa_si = svd_si.fit_transform(X_si)
lsa_ta = svd_ta.fit_transform(X_ta)

print(f"\nLSA variance explained — "
      f"EN: {svd_en.explained_variance_ratio_.sum():.3f} (150 components), "
      f"SI: {svd_si.explained_variance_ratio_.sum():.3f} (50 components), "
      f"TA: {svd_ta.explained_variance_ratio_.sum():.3f} (50 components)")

# ── 5. Expand to full matrix ──────────────────────────────────────────────────
def expand_sparse(block, mask_values, n_rows):
    full = lil_matrix((n_rows, block.shape[1]))
    for i, row_i in enumerate(np.where(mask_values)[0]):
        full[row_i] = block[i]
    return full.tocsr()

def expand_dense(block, mask_values, n_rows, n_cols):
    full = np.zeros((n_rows, n_cols))
    full[np.where(mask_values)[0]] = block
    return full

n = len(df_clean)
X_en_full   = expand_sparse(X_en, en_mask.values, n)
X_si_full   = expand_sparse(X_si, si_mask.values, n)
X_ta_full   = expand_sparse(X_ta, ta_mask.values, n)

lsa_en_full = expand_dense(lsa_en, en_mask.values, n, 150)
lsa_si_full = expand_dense(lsa_si, si_mask.values, n, 50)
lsa_ta_full = expand_dense(lsa_ta, ta_mask.values, n, 50)

scaler = StandardScaler()
X_hand = scaler.fit_transform(extract_handcrafted(df_clean))

X_clf     = hstack([csr_matrix(X_hand), X_en_full, X_si_full, X_ta_full])
X_reg_base = np.hstack([X_hand, lsa_en_full, lsa_si_full, lsa_ta_full])

# ── 6. Labels ─────────────────────────────────────────────────────────────────
le = LabelEncoder()
y  = le.fit_transform(df_clean["quality_label"])
s  = df_clean["overall_quality_score"].values.astype(float)

# ── 7. Split ──────────────────────────────────────────────────────────────────
idx = np.arange(n)
(X_clf_tr, X_clf_te,
 X_reg_tr,  X_reg_te,
 y_tr, y_te, s_tr, s_te,
 idx_tr, idx_te) = train_test_split(
    X_clf, X_reg_base, y, s, idx,
    test_size=0.2, random_state=42, stratify=y,
)

# ── 8. Classifier ─────────────────────────────────────────────────────────────
base_clf = RandomForestClassifier(
    n_estimators=500, max_depth=20, min_samples_leaf=5,
    max_features="sqrt", class_weight="balanced",
    random_state=42, n_jobs=-1,
)
clf = CalibratedClassifierCV(base_clf, method="isotonic", cv=5)
clf.fit(X_clf_tr, y_tr)

y_pred  = clf.predict(X_clf_te)
y_proba = clf.predict_proba(X_clf_te)
conf    = y_proba.max(axis=1)

print("\n── Classifier Results ───────────────────────────────────────────")
print(classification_report(y_te, y_pred, target_names=le.classes_))
print("Confusion matrix:")
print(pd.DataFrame(
    confusion_matrix(y_te, y_pred),
    index   = [f"actual_{c}" for c in le.classes_],
    columns = [f"pred_{c}"   for c in le.classes_],
))
print(f"\nMean confidence (all):     {conf.mean():.3f}")
print(f"Mean confidence (correct): {conf[y_pred == y_te].mean():.3f}")
print(f"Mean confidence (wrong):   {conf[y_pred != y_te].mean():.3f}")

# ── 9. CV ─────────────────────────────────────────────────────────────────────
cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cvs = cross_val_score(base_clf, X_clf, y, cv=cv, scoring="accuracy", n_jobs=-1)
print(f"\n5-fold CV accuracy: {cvs.mean():.3f} ± {cvs.std():.3f}")
print(f"Per-fold:           {np.round(cvs, 3)}")

# ── 10. Per-language accuracy ─────────────────────────────────────────────────
te_df = df_clean.iloc[idx_te].copy()
te_df["pred_label"] = le.inverse_transform(y_pred)
te_df["correct"]    = (y_pred == y_te)
print("\n── Per-language accuracy ────────────────────────────────────────")
print(te_df.groupby("language")["correct"].agg(["mean", "sum", "count"]).round(3))

# ── 11. Regressor — GBM + isotonic score anchoring ───────────────────────────
proba_tr = clf.predict_proba(X_clf_tr)
proba_te = clf.predict_proba(X_clf_te)

X_full_reg_tr = np.hstack([proba_tr, X_reg_tr])
X_full_reg_te = np.hstack([proba_te, X_reg_te])

reg = GradientBoostingRegressor(
    n_estimators=300, max_depth=4,
    learning_rate=0.05, subsample=0.8,
    random_state=42,
)
reg.fit(X_full_reg_tr, s_tr)
s_raw = np.clip(reg.predict(X_full_reg_te), 1.0, 5.0)

# ── 12. Score anchoring via isotonic post-correction ─────────────────────────
# The isotonic regressor learns a monotone mapping from raw GBM scores
# to true scores on the training set. This corrects systematic bias
# (e.g. the model always under-predicts score=5) and forces alignment
# with the classifier's probability ordering.
s_raw_tr = np.clip(reg.predict(X_full_reg_tr), 1.0, 5.0)
iso       = IsotonicRegression(y_min=1.0, y_max=5.0, out_of_bounds="clip")
iso.fit(s_raw_tr, s_tr)

s_pred = iso.predict(s_raw)

print("\n── Regressor Results (with isotonic anchoring) ──────────────────")
print(f"MAE : {mean_absolute_error(s_te, s_pred):.3f}")
print(f"R²  : {r2_score(s_te, s_pred):.3f}")
print(f"Rounded score match: {(np.round(s_pred) == s_te).mean():.3f}")

# Classifier / regressor agreement
def score_to_label(score):
    if score <= 2: return "low"
    if score <= 3: return "medium"
    return "high"

implied   = np.array([score_to_label(v) for v in s_pred])
clf_label = le.inverse_transform(y_pred)
agreement = (implied == clf_label).mean()
print(f"Classifier / regressor label agreement: {agreement:.3f}")

print("\n── Per-language regressor MAE ───────────────────────────────────")
te_df["pred_score"] = s_pred
te_df["true_score"] = s_te
for lang, grp in te_df.groupby("language"):
    mae = mean_absolute_error(grp["true_score"], grp["pred_score"])
    print(f"  {lang:8s}: MAE={mae:.3f}  n={len(grp)}")

# ── 13. Calibration reliability check ────────────────────────────────────────
# Bin confidence into deciles and check actual accuracy per bin.
# A well-calibrated model shows conf ≈ accuracy in each bin.
# This is what you include in your research paper.
print("\n── Calibration reliability (confidence bin → actual accuracy) ───")
bins        = np.linspace(0, 1, 11)
bin_indices = np.digitize(conf, bins) - 1
print(f"{'Conf bin':>12}  {'Accuracy':>8}  {'Count':>6}")
for b in range(10):
    mask = bin_indices == b
    if mask.sum() == 0:
        continue
    bin_acc = (y_pred[mask] == y_te[mask]).mean()
    bin_mid = (bins[b] + bins[b+1]) / 2
    print(f"  {bins[b]:.1f}–{bins[b+1]:.1f}    {bin_acc:.3f}     {mask.sum():>5}")

Rows: 5627
language
English    4600
Tamil       516
Sinhala     511
Name: count, dtype: int64

LSA variance explained — EN: 0.191 (150 components), SI: 0.561 (50 components), TA: 0.569 (50 components)

── Classifier Results ───────────────────────────────────────────
              precision    recall  f1-score   support

        high       0.80      0.75      0.78       394
         low       0.90      0.93      0.91       336
      medium       0.71      0.74      0.73       396

    accuracy                           0.80      1126
   macro avg       0.81      0.81      0.81      1126
weighted avg       0.80      0.80      0.80      1126

Confusion matrix:
               pred_high  pred_low  pred_medium
actual_high          296         1           97
actual_low             3       311           22
actual_medium         70        32          294

Mean confidence (all):     0.795
Mean confidence (correct): 0.834
Mean confidence (wrong):   0.642

5-fold CV accuracy: 0.784 ± 0.010
Per-fo

In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing           import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble                import RandomForestClassifier, GradientBoostingRegressor
from sklearn.calibration             import CalibratedClassifierCV
from sklearn.isotonic                import IsotonicRegression
from sklearn.model_selection         import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics                 import (classification_report, confusion_matrix,
                                             mean_absolute_error, r2_score)
from sklearn.decomposition           import TruncatedSVD
from scipy.sparse                    import hstack, csr_matrix, lil_matrix

# ── 1. Load ───────────────────────────────────────────────────────────────────
df = pd.read_csv("Donation_Appeal_MASTER_FINAL_v8_7100.csv")
df = df.dropna(subset=["appeal_text", "quality_label", "overall_quality_score"])
df["appeal_text"] = df["appeal_text"].astype(str).str.strip()
df = df[df["appeal_text"].str.len() > 10].reset_index(drop=True)

# ── 2. Downsample augmented non-English (40%) ─────────────────────────────────
AUGMENTED_ORIGINS = {
    "controlled_augmentation_native_reviewed",
    "controlled_augmentation_from_original_english",
    "AI_GENERATED_BOUNDARY_CASE",
    "AI_GENERATED_NOISY_CASE",
}
non_english_aug = df[
    (df["language"].isin(["Sinhala", "Tamil"])) &
    (df["data_origin"].isin(AUGMENTED_ORIGINS))
]
rest       = df[~df.index.isin(non_english_aug.index)]
aug_sample = non_english_aug.groupby(
    ["language", "quality_label"], group_keys=False
).apply(lambda g: g.sample(frac=0.40, random_state=42), include_groups=False)
df_clean = pd.concat([rest, df.loc[aug_sample.index]]).reset_index(drop=True)
print(f"Training corpus: {len(df_clean)} rows")

# ── 3. Handcrafted features ───────────────────────────────────────────────────
def extract_handcrafted(frame):
    t = frame["appeal_text"].astype(str)
    f = pd.DataFrame(index=frame.index)
    f["log_text_len"]      = np.log1p(t.str.len())
    f["log_word_count"]    = np.log1p(t.str.split().str.len())
    f["avg_word_len"]      = t.str.len() / (t.str.split().str.len() + 1)
    f["sentence_count"]    = np.log1p(t.str.count(r"[.!?।]") + 1)
    f["avg_sent_len"]      = t.str.split().str.len() / (t.str.count(r"[.!?།]") + 2)
    f["exclamation_count"] = np.log1p(t.str.count(r"!"))
    f["question_count"]    = np.log1p(t.str.count(r"\?"))
    f["digit_ratio"]       = t.str.count(r"\d") / (t.str.len() + 1)
    f["upper_ratio"]       = t.str.count(r"[A-Z]") / (t.str.len() + 1)
    f["has_currency"]      = t.str.contains(r"[$£€Rs\d]", regex=True).astype(int)
    f["has_donate_word"]   = t.str.contains(
        r"donat|contribut|help|support|give|fund|රුපියල්|දීමනා|நன்கொடை",
        case=False, regex=True
    ).astype(int)
    f["is_sinhala"]        = (frame["language"] == "Sinhala").astype(int)
    f["is_tamil"]          = (frame["language"] == "Tamil").astype(int)
    return f.fillna(0).values

# ── 4. Per-language TF-IDF + LSA ─────────────────────────────────────────────
en_mask = df_clean["language"] == "English"
si_mask = df_clean["language"] == "Sinhala"
ta_mask = df_clean["language"] == "Tamil"

def fit_tfidf(texts, analyzer, ngram_range, max_features):
    vec = TfidfVectorizer(
        analyzer=analyzer, ngram_range=ngram_range,
        max_features=max_features, sublinear_tf=True,
        strip_accents=None, min_df=3,
    )
    return vec, vec.fit_transform(texts)

tfidf_en, X_en = fit_tfidf(df_clean.loc[en_mask, "appeal_text"], "word",    (1, 2), 25_000)
tfidf_si, X_si = fit_tfidf(df_clean.loc[si_mask, "appeal_text"], "char_wb", (3, 5),  8_000)
tfidf_ta, X_ta = fit_tfidf(df_clean.loc[ta_mask, "appeal_text"], "char_wb", (3, 5),  8_000)

svd_en = TruncatedSVD(n_components=150, random_state=42)
svd_si = TruncatedSVD(n_components=50,  random_state=42)
svd_ta = TruncatedSVD(n_components=50,  random_state=42)

lsa_en = svd_en.fit_transform(X_en)
lsa_si = svd_si.fit_transform(X_si)
lsa_ta = svd_ta.fit_transform(X_ta)

# ── 5. Expand to full-corpus matrices ─────────────────────────────────────────
def expand_sparse(block, mask_values, n_rows):
    full = lil_matrix((n_rows, block.shape[1]))
    for i, row_i in enumerate(np.where(mask_values)[0]):
        full[row_i] = block[i]
    return full.tocsr()

def expand_dense(block, mask_values, n_rows, n_cols):
    full = np.zeros((n_rows, n_cols))
    full[np.where(mask_values)[0]] = block
    return full

n = len(df_clean)
X_en_full   = expand_sparse(X_en, en_mask.values, n)
X_si_full   = expand_sparse(X_si, si_mask.values, n)
X_ta_full   = expand_sparse(X_ta, ta_mask.values, n)
lsa_en_full = expand_dense(lsa_en, en_mask.values, n, 150)
lsa_si_full = expand_dense(lsa_si, si_mask.values, n, 50)
lsa_ta_full = expand_dense(lsa_ta, ta_mask.values, n, 50)

scaler     = StandardScaler()
X_hand     = scaler.fit_transform(extract_handcrafted(df_clean))
X_clf      = hstack([csr_matrix(X_hand), X_en_full, X_si_full, X_ta_full])
X_reg_base = np.hstack([X_hand, lsa_en_full, lsa_si_full, lsa_ta_full])

# ── 6. Labels ─────────────────────────────────────────────────────────────────
le = LabelEncoder()
y  = le.fit_transform(df_clean["quality_label"])
s  = df_clean["overall_quality_score"].values.astype(float)
print("Label encoding:", dict(zip(le.classes_, le.transform(le.classes_))))

# ── 7. Split ──────────────────────────────────────────────────────────────────
idx = np.arange(n)
(X_clf_tr, X_clf_te,
 X_reg_tr,  X_reg_te,
 y_tr, y_te, s_tr, s_te,
 idx_tr, idx_te) = train_test_split(
    X_clf, X_reg_base, y, s, idx,
    test_size=0.2, random_state=42, stratify=y,
)

# ── 8. Classifier ─────────────────────────────────────────────────────────────
base_clf = RandomForestClassifier(
    n_estimators=500, max_depth=20, min_samples_leaf=5,
    max_features="sqrt", class_weight="balanced",
    random_state=42, n_jobs=-1,
)
clf = CalibratedClassifierCV(base_clf, method="isotonic", cv=5)
clf.fit(X_clf_tr, y_tr)

y_pred  = clf.predict(X_clf_te)
y_proba = clf.predict_proba(X_clf_te)
conf    = y_proba.max(axis=1)

print("\n── Classifier Results ───────────────────────────────────────────")
print(classification_report(y_te, y_pred, target_names=le.classes_))
print(pd.DataFrame(
    confusion_matrix(y_te, y_pred),
    index   = [f"actual_{c}" for c in le.classes_],
    columns = [f"pred_{c}"   for c in le.classes_],
))
print(f"\nMean confidence (all):     {conf.mean():.3f}")
print(f"Mean confidence (correct): {conf[y_pred == y_te].mean():.3f}")
print(f"Mean confidence (wrong):   {conf[y_pred != y_te].mean():.3f}")

cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cvs = cross_val_score(base_clf, X_clf, y, cv=cv, scoring="accuracy", n_jobs=-1)
print(f"\n5-fold CV: {cvs.mean():.3f} ± {cvs.std():.3f}  {np.round(cvs, 3)}")

te_df = df_clean.iloc[idx_te].copy()
te_df["pred_label"] = le.inverse_transform(y_pred)
te_df["correct"]    = (y_pred == y_te)
print("\n── Per-language accuracy ────────────────────────────────────────")
print(te_df.groupby("language")["correct"].agg(["mean", "sum", "count"]).round(3))

# ── 9. Regressor ──────────────────────────────────────────────────────────────
proba_tr = clf.predict_proba(X_clf_tr)
proba_te = clf.predict_proba(X_clf_te)

X_full_reg_tr = np.hstack([proba_tr, X_reg_tr])
X_full_reg_te = np.hstack([proba_te, X_reg_te])

reg = GradientBoostingRegressor(
    n_estimators=300, max_depth=4,
    learning_rate=0.05, subsample=0.8,
    random_state=42,
)
reg.fit(X_full_reg_tr, s_tr)

s_raw_tr = np.clip(reg.predict(X_full_reg_tr), 1.0, 5.0)
s_raw_te = np.clip(reg.predict(X_full_reg_te), 1.0, 5.0)

iso = IsotonicRegression(y_min=1.0, y_max=5.0, out_of_bounds="clip")
iso.fit(s_raw_tr, s_tr)
s_pred = iso.predict(s_raw_te)

# ── 10. Soft boundary agreement ───────────────────────────────────────────────
# Instead of hard cutoffs, use classifier probability to resolve boundary cases.
# If the raw score is within 0.4 of a boundary (2.0 or 3.0), defer to the
# classifier's argmax label rather than the score bucket.
BOUNDARY_TOLERANCE = 0.4
label_order = {name: i for i, name in enumerate(le.classes_)}  # alphabetical idx

def resolve_label(score, proba_row, classes):
    """
    Return the label that both score and classifier agree on.
    At boundaries, defer to classifier confidence.
    """
    # Raw score bucket
    if score <= 2.0:
        score_label = "low"
    elif score <= 3.0:
        score_label = "medium"
    else:
        score_label = "high"

    clf_label = classes[np.argmax(proba_row)]

    if score_label == clf_label:
        return score_label

    # In boundary zone — defer to classifier
    near_low_boundary    = abs(score - 2.0) < BOUNDARY_TOLERANCE
    near_medium_boundary = abs(score - 3.0) < BOUNDARY_TOLERANCE
    if near_low_boundary or near_medium_boundary:
        return clf_label

    return score_label

resolved_labels = np.array([
    resolve_label(sc, pr, le.classes_)
    for sc, pr in zip(s_pred, y_proba)
])
clf_labels = le.inverse_transform(y_pred)
agreement  = (resolved_labels == clf_labels).mean()

print("\n── Regressor Results ────────────────────────────────────────────")
print(f"MAE:                 {mean_absolute_error(s_te, s_pred):.3f}")
print(f"R²:                  {r2_score(s_te, s_pred):.3f}")
print(f"Rounded score match: {(np.round(s_pred) == s_te).mean():.3f}")
print(f"Clf/reg agreement (soft boundary): {agreement:.3f}")

te_df["pred_score"] = s_pred
te_df["true_score"] = s_te
print("\n── Per-language MAE ─────────────────────────────────────────────")
for lang, grp in te_df.groupby("language"):
    mae = mean_absolute_error(grp["true_score"], grp["pred_score"])
    print(f"  {lang:8s}: MAE={mae:.3f}  n={len(grp)}")

print("\n── Calibration reliability ──────────────────────────────────────")
bins = np.linspace(0, 1, 11)
bin_indices = np.digitize(conf, bins) - 1
print(f"{'Conf bin':>12}  {'Accuracy':>8}  {'Count':>6}")
for b in range(10):
    mask = bin_indices == b
    if mask.sum() == 0:
        continue
    print(f"  {bins[b]:.1f}–{bins[b+1]:.1f}    "
          f"{(y_pred[mask] == y_te[mask]).mean():.3f}     {mask.sum():>5}")

# ── 11. Gate: only save if targets are met ────────────────────────────────────
targets_met = (
    cvs.mean()   >= 0.78  and
    cvs.std()    <= 0.015 and
    agreement    >= 0.82  and
    r2_score(s_te, s_pred) >= 0.74
)

print(f"\n── Artifact save gate ───────────────────────────────────────────")
print(f"CV mean  >= 0.78 : {cvs.mean():.3f}  {'✓' if cvs.mean() >= 0.78 else '✗'}")
print(f"CV std   <= 0.015: {cvs.std():.3f}  {'✓' if cvs.std() <= 0.015 else '✗'}")
print(f"Agreement>= 0.82 : {agreement:.3f}  {'✓' if agreement >= 0.82 else '✗'}")
print(f"R²       >= 0.74 : {r2_score(s_te, s_pred):.3f}  {'✓' if r2_score(s_te, s_pred) >= 0.74 else '✗'}")

if targets_met:
    print("\nAll targets met — saving artifacts...")

    # Bundle all inference components into a single config dict
    # so quality_service.py loads one file for metadata
    model_config = {
        "label_classes"      : le.classes_.tolist(),
        "boundary_tolerance" : BOUNDARY_TOLERANCE,
        "lsa_components"     : {"en": 150, "si": 50, "ta": 50},
        "scaler_feature_names": [
            "log_text_len", "log_word_count", "avg_word_len",
            "sentence_count", "avg_sent_len", "exclamation_count",
            "question_count", "digit_ratio", "upper_ratio",
            "has_currency", "has_donate_word", "is_sinhala", "is_tamil",
        ],
    }

    joblib.dump(tfidf_en,     "appeal_quality_vectorizer.joblib")      # English TF-IDF
    joblib.dump(tfidf_si,     "appeal_quality_vectorizer_si.joblib")   # Sinhala TF-IDF
    joblib.dump(tfidf_ta,     "appeal_quality_vectorizer_ta.joblib")   # Tamil TF-IDF
    joblib.dump(svd_en,       "appeal_quality_svd_en.joblib")          # English LSA
    joblib.dump(svd_si,       "appeal_quality_svd_si.joblib")          # Sinhala LSA
    joblib.dump(svd_ta,       "appeal_quality_svd_ta.joblib")          # Tamil LSA
    joblib.dump(scaler,       "appeal_quality_scaler.joblib")          # handcrafted scaler
    joblib.dump(clf,          "appeal_quality_classifier.joblib")      # calibrated RF
    joblib.dump(le,           "appeal_quality_label_encoder.joblib")   # label encoder
    joblib.dump(reg,          "appeal_quality_regressor.joblib")       # GBM regressor
    joblib.dump(iso,          "appeal_quality_isotonic.joblib")        # isotonic corrector
    joblib.dump(model_config, "appeal_quality_config.joblib")          # inference metadata

    print("Saved 12 artifacts:")
    for name in [
        "appeal_quality_vectorizer.joblib",
        "appeal_quality_vectorizer_si.joblib",
        "appeal_quality_vectorizer_ta.joblib",
        "appeal_quality_svd_en.joblib",
        "appeal_quality_svd_si.joblib",
        "appeal_quality_svd_ta.joblib",
        "appeal_quality_scaler.joblib",
        "appeal_quality_classifier.joblib",
        "appeal_quality_label_encoder.joblib",
        "appeal_quality_regressor.joblib",
        "appeal_quality_isotonic.joblib",
        "appeal_quality_config.joblib",
    ]:
        print(f"  {name}")
else:
    print("\nTargets not met — do not save. Paste output for next step.")

Training corpus: 5627 rows
Label encoding: {'high': np.int64(0), 'low': np.int64(1), 'medium': np.int64(2)}

── Classifier Results ───────────────────────────────────────────
              precision    recall  f1-score   support

        high       0.80      0.75      0.78       394
         low       0.90      0.93      0.91       336
      medium       0.71      0.74      0.73       396

    accuracy                           0.80      1126
   macro avg       0.81      0.81      0.81      1126
weighted avg       0.80      0.80      0.80      1126

               pred_high  pred_low  pred_medium
actual_high          296         1           97
actual_low             3       311           22
actual_medium         70        32          294

Mean confidence (all):     0.795
Mean confidence (correct): 0.834
Mean confidence (wrong):   0.642

5-fold CV: 0.784 ± 0.010  [0.768 0.775 0.795 0.791 0.791]

── Per-language accuracy ────────────────────────────────────────
           mean  sum  coun

In [ ]:
from google.colab import files

artifact_names = [
    "appeal_quality_vectorizer.joblib",
    "appeal_quality_vectorizer_si.joblib",
    "appeal_quality_vectorizer_ta.joblib",
    "appeal_quality_svd_en.joblib",
    "appeal_quality_svd_si.joblib",
    "appeal_quality_svd_ta.joblib",
    "appeal_quality_scaler.joblib",
    "appeal_quality_classifier.joblib",
    "appeal_quality_label_encoder.joblib",
    "appeal_quality_regressor.joblib",
    "appeal_quality_isotonic.joblib",
    "appeal_quality_config.joblib",
]

for name in artifact_names:
    files.download(name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Retrain retrain the GBR from scratch in a clean numpy==1.26.4**

In [4]:
!sudo apt-get install -y python3.10 python3.10-dev python3.10-distutils -q
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3.10
!python3.10 -m pip install numpy==1.26.4 scikit-learn joblib pandas scipy --only-binary=:all: -q

Reading package lists...
Building dependency tree...
Reading state information...
python3-distutils is already the newest version (3.10.8-1~22.04).
python3-distutils set to manually installed.
python3.10 is already the newest version (3.10.12-1~22.04.16).
python3.10 set to manually installed.
The following NEW packages will be installed:
  python3.10-dev
0 upgraded, 1 newly installed, 0 to remove and 1 not upgraded.
Need to get 508 kB of archives.
After this operation, 523 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3.10-dev amd64 3.10.12-1~22.04.16 [508 kB]
Fetched 508 kB in 1s (611 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requ

In [5]:
!python3.10 -m pip install numpy==1.26.4 scikit-learn joblib pandas scipy --only-binary=:all: -q

In [7]:
!python3.10 -c "import numpy; print(numpy.__version__)"
!python3.10 -c "import sklearn; print(sklearn.__version__)"

1.26.4
1.7.2


In [10]:
!python3.10 -m pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [11]:
!python3.10 -c "import spacy; nlp = spacy.load('en_core_sci_sm'); print('scispaCy OK')"

/usr/local/lib/python3.10/dist-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]
scispaCy OK


**Full training**

In [12]:
%%writefile train_quality.py

import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing           import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble                import RandomForestClassifier, GradientBoostingRegressor
from sklearn.calibration             import CalibratedClassifierCV
from sklearn.isotonic                import IsotonicRegression
from sklearn.model_selection         import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics                 import (classification_report, confusion_matrix,
                                             mean_absolute_error, r2_score)
from sklearn.decomposition           import TruncatedSVD
from scipy.sparse                    import hstack, csr_matrix, lil_matrix

print(f"numpy : {np.__version__}")

# ── 1. Load ───────────────────────────────────────────────────────────────────
df = pd.read_csv("Donation_Appeal_MASTER_FINAL_v8_7100.csv")
df = df.dropna(subset=["appeal_text", "quality_label", "overall_quality_score"])
df["appeal_text"] = df["appeal_text"].astype(str).str.strip()
df = df[df["appeal_text"].str.len() > 10].reset_index(drop=True)

# ── 2. Downsample augmented non-English (40%) ─────────────────────────────────
AUGMENTED_ORIGINS = {
    "controlled_augmentation_native_reviewed",
    "controlled_augmentation_from_original_english",
    "AI_GENERATED_BOUNDARY_CASE",
    "AI_GENERATED_NOISY_CASE",
}
non_english_aug = df[
    (df["language"].isin(["Sinhala", "Tamil"])) &
    (df["data_origin"].isin(AUGMENTED_ORIGINS))
]
rest       = df[~df.index.isin(non_english_aug.index)]
aug_sample = non_english_aug.groupby(
    ["language", "quality_label"], group_keys=False
).apply(lambda g: g.sample(frac=0.40, random_state=42), include_groups=False)
df_clean = pd.concat([rest, df.loc[aug_sample.index]]).reset_index(drop=True)
print(f"Training corpus: {len(df_clean)} rows")
print(df_clean["language"].value_counts())

# ── 3. Handcrafted features ───────────────────────────────────────────────────
def extract_handcrafted(frame):
    t = frame["appeal_text"].astype(str)
    f = pd.DataFrame(index=frame.index)
    f["log_text_len"]      = np.log1p(t.str.len())
    f["log_word_count"]    = np.log1p(t.str.split().str.len())
    f["avg_word_len"]      = t.str.len() / (t.str.split().str.len() + 1)
    f["sentence_count"]    = np.log1p(t.str.count(r"[.!?।]") + 1)
    f["avg_sent_len"]      = t.str.split().str.len() / (t.str.count(r"[.!?།]") + 2)
    f["exclamation_count"] = np.log1p(t.str.count(r"!"))
    f["question_count"]    = np.log1p(t.str.count(r"\?"))
    f["digit_ratio"]       = t.str.count(r"\d") / (t.str.len() + 1)
    f["upper_ratio"]       = t.str.count(r"[A-Z]") / (t.str.len() + 1)
    f["has_currency"]      = t.str.contains(r"[$£€Rs\d]", regex=True).astype(int)
    f["has_donate_word"]   = t.str.contains(
        r"donat|contribut|help|support|give|fund|රුපියල්|දීමනා|நன்கொடை",
        case=False, regex=True
    ).astype(int)
    f["is_sinhala"]        = (frame["language"] == "Sinhala").astype(int)
    f["is_tamil"]          = (frame["language"] == "Tamil").astype(int)
    return f.fillna(0).values

# ── 4. Per-language TF-IDF + LSA ─────────────────────────────────────────────
en_mask = df_clean["language"] == "English"
si_mask = df_clean["language"] == "Sinhala"
ta_mask = df_clean["language"] == "Tamil"

def fit_tfidf(texts, analyzer, ngram_range, max_features):
    vec = TfidfVectorizer(
        analyzer=analyzer, ngram_range=ngram_range,
        max_features=max_features, sublinear_tf=True,
        strip_accents=None, min_df=3,
    )
    return vec, vec.fit_transform(texts)

tfidf_en, X_en = fit_tfidf(df_clean.loc[en_mask, "appeal_text"], "word",    (1, 2), 25_000)
tfidf_si, X_si = fit_tfidf(df_clean.loc[si_mask, "appeal_text"], "char_wb", (3, 5),  8_000)
tfidf_ta, X_ta = fit_tfidf(df_clean.loc[ta_mask, "appeal_text"], "char_wb", (3, 5),  8_000)

svd_en = TruncatedSVD(n_components=150, random_state=42)
svd_si = TruncatedSVD(n_components=50,  random_state=42)
svd_ta = TruncatedSVD(n_components=50,  random_state=42)

lsa_en = svd_en.fit_transform(X_en)
lsa_si = svd_si.fit_transform(X_si)
lsa_ta = svd_ta.fit_transform(X_ta)

print(f"\nLSA variance — EN: {svd_en.explained_variance_ratio_.sum():.3f}, "
      f"SI: {svd_si.explained_variance_ratio_.sum():.3f}, "
      f"TA: {svd_ta.explained_variance_ratio_.sum():.3f}")

# ── 5. Expand to full-corpus matrices ─────────────────────────────────────────
def expand_sparse(block, mask_values, n_rows):
    full = lil_matrix((n_rows, block.shape[1]))
    for i, row_i in enumerate(np.where(mask_values)[0]):
        full[row_i] = block[i]
    return full.tocsr()

def expand_dense(block, mask_values, n_rows, n_cols):
    full = np.zeros((n_rows, n_cols))
    full[np.where(mask_values)[0]] = block
    return full

n = len(df_clean)
X_en_full   = expand_sparse(X_en, en_mask.values, n)
X_si_full   = expand_sparse(X_si, si_mask.values, n)
X_ta_full   = expand_sparse(X_ta, ta_mask.values, n)
lsa_en_full = expand_dense(lsa_en, en_mask.values, n, 150)
lsa_si_full = expand_dense(lsa_si, si_mask.values, n, 50)
lsa_ta_full = expand_dense(lsa_ta, ta_mask.values, n, 50)

scaler     = StandardScaler()
X_hand     = scaler.fit_transform(extract_handcrafted(df_clean))
X_clf      = hstack([csr_matrix(X_hand), X_en_full, X_si_full, X_ta_full])
X_reg_base = np.hstack([X_hand, lsa_en_full, lsa_si_full, lsa_ta_full])

print(f"Classifier input : {X_clf.shape}")
print(f"Regressor base   : {X_reg_base.shape}")

# ── 6. Labels ─────────────────────────────────────────────────────────────────
le = LabelEncoder()
y  = le.fit_transform(df_clean["quality_label"])
s  = df_clean["overall_quality_score"].values.astype(float)
print("Label encoding:", dict(zip(le.classes_, le.transform(le.classes_))))

# ── 7. Split ──────────────────────────────────────────────────────────────────
idx = np.arange(n)
(X_clf_tr, X_clf_te,
 X_reg_tr,  X_reg_te,
 y_tr, y_te, s_tr, s_te,
 idx_tr, idx_te) = train_test_split(
    X_clf, X_reg_base, y, s, idx,
    test_size=0.2, random_state=42, stratify=y,
)
print(f"Train: {X_clf_tr.shape[0]}  |  Test: {X_clf_te.shape[0]}")

# ── 8. Classifier ─────────────────────────────────────────────────────────────
base_clf = RandomForestClassifier(
    n_estimators=500, max_depth=20, min_samples_leaf=5,
    max_features="sqrt", class_weight="balanced",
    random_state=42, n_jobs=-1,
)
clf = CalibratedClassifierCV(base_clf, method="isotonic", cv=5)
clf.fit(X_clf_tr, y_tr)

y_pred  = clf.predict(X_clf_te)
y_proba = clf.predict_proba(X_clf_te)
conf    = y_proba.max(axis=1)

print("\n── Classifier Results ───────────────────────────────────────────")
print(classification_report(y_te, y_pred, target_names=le.classes_))
print(pd.DataFrame(
    confusion_matrix(y_te, y_pred),
    index   = [f"actual_{c}" for c in le.classes_],
    columns = [f"pred_{c}"   for c in le.classes_],
))
print(f"\nMean confidence (all):     {conf.mean():.3f}")
print(f"Mean confidence (correct): {conf[y_pred == y_te].mean():.3f}")
print(f"Mean confidence (wrong):   {conf[y_pred != y_te].mean():.3f}")

cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cvs = cross_val_score(base_clf, X_clf, y, cv=cv, scoring="accuracy", n_jobs=-1)
print(f"\n5-fold CV: {cvs.mean():.3f} ± {cvs.std():.3f}  {np.round(cvs, 3)}")

te_df = df_clean.iloc[idx_te].copy()
te_df["pred_label"] = le.inverse_transform(y_pred)
te_df["correct"]    = (y_pred == y_te)
print("\n── Per-language accuracy ────────────────────────────────────────")
print(te_df.groupby("language")["correct"].agg(["mean", "sum", "count"]).round(3))

# ── 9. Regressor ──────────────────────────────────────────────────────────────
proba_tr = clf.predict_proba(X_clf_tr)
proba_te = clf.predict_proba(X_clf_te)

X_full_reg_tr = np.hstack([proba_tr, X_reg_tr])
X_full_reg_te = np.hstack([proba_te, X_reg_te])

reg = GradientBoostingRegressor(
    n_estimators=300, max_depth=4,
    learning_rate=0.05, subsample=0.8,
    random_state=42,
)
reg.fit(X_full_reg_tr, s_tr)

s_raw_tr = np.clip(reg.predict(X_full_reg_tr), 1.0, 5.0)
s_raw_te = np.clip(reg.predict(X_full_reg_te), 1.0, 5.0)

iso = IsotonicRegression(y_min=1.0, y_max=5.0, out_of_bounds="clip")
iso.fit(s_raw_tr, s_tr)
s_pred = iso.predict(s_raw_te)

print("\n── Regressor Results ────────────────────────────────────────────")
r2  = r2_score(s_te, s_pred)
mae = mean_absolute_error(s_te, s_pred)
print(f"MAE : {mae:.3f}")
print(f"R²  : {r2:.3f}")
print(f"Rounded score match: {(np.round(s_pred) == s_te).mean():.3f}")

te_df["pred_score"] = s_pred
te_df["true_score"] = s_te
print("\n── Per-language MAE ─────────────────────────────────────────────")
for lang, grp in te_df.groupby("language"):
    print(f"  {lang:8s}: MAE={mean_absolute_error(grp['true_score'], grp['pred_score']):.3f}  n={len(grp)}")

# ── 10. Classifier / regressor agreement ──────────────────────────────────────
clf_reg_agreement = (
    np.array(["low" if v<=2 else "medium" if v<=3 else "high" for v in s_pred]) ==
    le.inverse_transform(y_pred)
).mean()

# ── 11. Save gate ─────────────────────────────────────────────────────────────
print("\n── Save gate ────────────────────────────────────────────────────")
print(f"CV mean  >= 0.78 : {cvs.mean():.3f}  {'✓' if cvs.mean() >= 0.78 else '✗'}")
print(f"CV std   <= 0.015: {cvs.std():.4f} {'✓' if cvs.std() <= 0.015 else '✗'}")
print(f"R²       >= 0.74 : {r2:.3f}  {'✓' if r2 >= 0.74 else '✗'}")
print(f"Agreement>= 0.82 : {clf_reg_agreement:.3f}  {'✓' if clf_reg_agreement >= 0.82 else '✗'}")

BOUNDARY_TOLERANCE = 0.4
model_config = {
    "label_classes"       : le.classes_.tolist(),
    "boundary_tolerance"  : BOUNDARY_TOLERANCE,
    "lsa_components"      : {"en": 150, "si": 50, "ta": 50},
    "scaler_feature_names": [
        "log_text_len", "log_word_count", "avg_word_len",
        "sentence_count", "avg_sent_len", "exclamation_count",
        "question_count", "digit_ratio", "upper_ratio",
        "has_currency", "has_donate_word", "is_sinhala", "is_tamil",
    ],
}

joblib.dump(tfidf_en,     "appeal_quality_vectorizer.joblib")
joblib.dump(tfidf_si,     "appeal_quality_vectorizer_si.joblib")
joblib.dump(tfidf_ta,     "appeal_quality_vectorizer_ta.joblib")
joblib.dump(svd_en,       "appeal_quality_svd_en.joblib")
joblib.dump(svd_si,       "appeal_quality_svd_si.joblib")
joblib.dump(svd_ta,       "appeal_quality_svd_ta.joblib")
joblib.dump(scaler,       "appeal_quality_scaler.joblib")
joblib.dump(clf,          "appeal_quality_classifier.joblib")
joblib.dump(le,           "appeal_quality_label_encoder.joblib")
joblib.dump(reg,          "appeal_quality_regressor.joblib")
joblib.dump(iso,          "appeal_quality_isotonic.joblib")
joblib.dump(model_config, "appeal_quality_config.joblib")

print(f"\nAll 12 artifacts saved — numpy {np.__version__}")

Writing train_quality.py


In [13]:
!python3.10 train_quality.py

numpy : 1.26.4
Traceback (most recent call last):
  File "/content/train_quality.py", line 20, in <module>
    df = pd.read_csv("Donation_Appeal_MASTER_FINAL_v8_7100.csv")
  File "/usr/local/lib/python3.10/dist-packages/pandas/io/parsers/readers.py", line 1026, in read_csv
    return _read(filepath_or_buffer, kwds)
  File "/usr/local/lib/python3.10/dist-packages/pandas/io/parsers/readers.py", line 620, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
  File "/usr/local/lib/python3.10/dist-packages/pandas/io/parsers/readers.py", line 1620, in __init__
    self._engine = self._make_engine(f, self.engine)
  File "/usr/local/lib/python3.10/dist-packages/pandas/io/parsers/readers.py", line 1880, in _make_engine
    self.handles = get_handle(
  File "/usr/local/lib/python3.10/dist-packages/pandas/io/common.py", line 873, in get_handle
    handle = open(
FileNotFoundError: [Errno 2] No such file or directory: 'Donation_Appeal_MASTER_FINAL_v8_7100.csv'


In [14]:
from google.colab import files
uploaded = files.upload()  # upload Donation_Appeal_MASTER_FINAL_v8_7100.csv

Saving Donation_Appeal_MASTER_FINAL_v8_7100.csv to Donation_Appeal_MASTER_FINAL_v8_7100.csv


In [15]:
!python3.10 train_quality.py

numpy : 1.26.4
Training corpus: 5627 rows
language
English    4600
Tamil       516
Sinhala     511
Name: count, dtype: int64

LSA variance — EN: 0.191, SI: 0.561, TA: 0.569
Classifier input : (5627, 39225)
Regressor base   : (5627, 263)
Label encoding: {'high': 0, 'low': 1, 'medium': 2}
Train: 4501  |  Test: 1126

── Classifier Results ───────────────────────────────────────────
              precision    recall  f1-score   support

        high       0.79      0.78      0.78       394
         low       0.91      0.93      0.92       336
      medium       0.73      0.73      0.73       396

    accuracy                           0.80      1126
   macro avg       0.81      0.81      0.81      1126
weighted avg       0.80      0.80      0.80      1126

               pred_high  ...  pred_medium
actual_high          306  ...           87
actual_low             3  ...           22
actual_medium         77  ...          288

[3 rows x 3 columns]

Mean confidence (all):     0.793
Mean conf

In [16]:
%%writefile fix_agreement.py
import numpy as np
import joblib
from sklearn.metrics import r2_score, mean_absolute_error

# Load what was just saved
clf  = joblib.load("appeal_quality_classifier.joblib")
reg  = joblib.load("appeal_quality_regressor.joblib")
iso  = joblib.load("appeal_quality_isotonic.joblib")
le   = joblib.load("appeal_quality_label_encoder.joblib")
cfg  = joblib.load("appeal_quality_config.joblib")

# We need X_clf_te, X_reg_te, y_te, s_te from the training run
# Re-derive them with the same split
import pandas as pd
from sklearn.preprocessing           import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition           import TruncatedSVD
from sklearn.model_selection         import train_test_split
from scipy.sparse                    import hstack, csr_matrix, lil_matrix

df = pd.read_csv("Donation_Appeal_MASTER_FINAL_v8_7100.csv")
df = df.dropna(subset=["appeal_text", "quality_label", "overall_quality_score"])
df["appeal_text"] = df["appeal_text"].astype(str).str.strip()
df = df[df["appeal_text"].str.len() > 10].reset_index(drop=True)

AUGMENTED_ORIGINS = {
    "controlled_augmentation_native_reviewed",
    "controlled_augmentation_from_original_english",
    "AI_GENERATED_BOUNDARY_CASE",
    "AI_GENERATED_NOISY_CASE",
}
non_english_aug = df[
    (df["language"].isin(["Sinhala", "Tamil"])) &
    (df["data_origin"].isin(AUGMENTED_ORIGINS))
]
rest       = df[~df.index.isin(non_english_aug.index)]
aug_sample = non_english_aug.groupby(
    ["language", "quality_label"], group_keys=False
).apply(lambda g: g.sample(frac=0.40, random_state=42), include_groups=False)
df_clean = pd.concat([rest, df.loc[aug_sample.index]]).reset_index(drop=True)

def extract_handcrafted(frame):
    t = frame["appeal_text"].astype(str)
    f = pd.DataFrame(index=frame.index)
    f["log_text_len"]      = np.log1p(t.str.len())
    f["log_word_count"]    = np.log1p(t.str.split().str.len())
    f["avg_word_len"]      = t.str.len() / (t.str.split().str.len() + 1)
    f["sentence_count"]    = np.log1p(t.str.count(r"[.!?।]") + 1)
    f["avg_sent_len"]      = t.str.split().str.len() / (t.str.count(r"[.!?།]") + 2)
    f["exclamation_count"] = np.log1p(t.str.count(r"!"))
    f["question_count"]    = np.log1p(t.str.count(r"\?"))
    f["digit_ratio"]       = t.str.count(r"\d") / (t.str.len() + 1)
    f["upper_ratio"]       = t.str.count(r"[A-Z]") / (t.str.len() + 1)
    f["has_currency"]      = t.str.contains(r"[$£€Rs\d]", regex=True).astype(int)
    f["has_donate_word"]   = t.str.contains(
        r"donat|contribut|help|support|give|fund|රුපියල්|දීමනා|நன்கொடை",
        case=False, regex=True
    ).astype(int)
    f["is_sinhala"]        = (frame["language"] == "Sinhala").astype(int)
    f["is_tamil"]          = (frame["language"] == "Tamil").astype(int)
    return f.fillna(0).values

tfidf_en = joblib.load("appeal_quality_vectorizer.joblib")
tfidf_si = joblib.load("appeal_quality_vectorizer_si.joblib")
tfidf_ta = joblib.load("appeal_quality_vectorizer_ta.joblib")
svd_en   = joblib.load("appeal_quality_svd_en.joblib")
svd_si   = joblib.load("appeal_quality_svd_si.joblib")
svd_ta   = joblib.load("appeal_quality_svd_ta.joblib")
scaler   = joblib.load("appeal_quality_scaler.joblib")

en_mask = df_clean["language"] == "English"
si_mask = df_clean["language"] == "Sinhala"
ta_mask = df_clean["language"] == "Tamil"

def expand_sparse(block, mask_values, n_rows):
    full = lil_matrix((n_rows, block.shape[1]))
    for i, row_i in enumerate(np.where(mask_values)[0]):
        full[row_i] = block[i]
    return full.tocsr()

def expand_dense(block, mask_values, n_rows, n_cols):
    full = np.zeros((n_rows, n_cols))
    full[np.where(mask_values)[0]] = block
    return full

n = len(df_clean)
X_en_full   = expand_sparse(tfidf_en.transform(df_clean.loc[en_mask, "appeal_text"]), en_mask.values, n)
X_si_full   = expand_sparse(tfidf_si.transform(df_clean.loc[si_mask, "appeal_text"]), si_mask.values, n)
X_ta_full   = expand_sparse(tfidf_ta.transform(df_clean.loc[ta_mask, "appeal_text"]), ta_mask.values, n)

lsa_en_full = expand_dense(svd_en.transform(tfidf_en.transform(df_clean.loc[en_mask, "appeal_text"])), en_mask.values, n, 150)
lsa_si_full = expand_dense(svd_si.transform(tfidf_si.transform(df_clean.loc[si_mask, "appeal_text"])), si_mask.values, n, 50)
lsa_ta_full = expand_dense(svd_ta.transform(tfidf_ta.transform(df_clean.loc[ta_mask, "appeal_text"])), ta_mask.values, n, 50)

X_hand     = scaler.transform(extract_handcrafted(df_clean))
X_clf      = hstack([csr_matrix(X_hand), X_en_full, X_si_full, X_ta_full])
X_reg_base = np.hstack([X_hand, lsa_en_full, lsa_si_full, lsa_ta_full])

le2 = LabelEncoder()
y   = le2.fit_transform(df_clean["quality_label"])
s   = df_clean["overall_quality_score"].values.astype(float)

idx = np.arange(n)
(X_clf_tr, X_clf_te,
 X_reg_tr,  X_reg_te,
 y_tr, y_te, s_tr, s_te,
 idx_tr, idx_te) = train_test_split(
    X_clf, X_reg_base, y, s, idx,
    test_size=0.2, random_state=42, stratify=y,
)

y_pred  = clf.predict(X_clf_te)
y_proba = clf.predict_proba(X_clf_te)
proba_te = y_proba

s_raw_te = np.clip(reg.predict(np.hstack([proba_te, X_reg_te])), 1.0, 5.0)
s_pred   = iso.predict(s_raw_te)

# Test multiple tolerance values
print("Tolerance sweep — finding best agreement without breaking classifier accuracy:")
print(f"{'Tolerance':>10}  {'Agreement':>10}  {'MAE':>6}  {'R²':>6}")
for tol in [0.0, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4]:
    def resolve(score, proba_row):
        if score <= 2.0:   sl = "low"
        elif score <= 3.0: sl = "medium"
        else:              sl = "high"
        cl = le.classes_[np.argmax(proba_row)]
        if sl == cl: return sl
        if abs(score - 2.0) < tol or abs(score - 3.0) < tol:
            return cl
        return sl

    resolved = np.array([resolve(sc, pr) for sc, pr in zip(s_pred, y_proba)])
    clf_labels = le.inverse_transform(y_pred)
    agreement  = (resolved == clf_labels).mean()
    print(f"  {tol:>8.2f}  {agreement:>10.3f}  {mean_absolute_error(s_te, s_pred):>6.3f}  {r2_score(s_te, s_pred):>6.3f}")

Writing fix_agreement.py


In [17]:
!python3.10 fix_agreement.py

Tolerance sweep — finding best agreement without breaking classifier accuracy:
 Tolerance   Agreement     MAE      R²
      0.00       0.795   0.333   0.755
      0.10       0.862   0.333   0.755
      0.15       0.865   0.333   0.755
      0.20       0.865   0.333   0.755
      0.25       0.878   0.333   0.755
      0.30       0.882   0.333   0.755
      0.35       0.885   0.333   0.755
      0.40       0.887   0.333   0.755


In [18]:
%%writefile update_config.py
import joblib

cfg = joblib.load("appeal_quality_config.joblib")
cfg["boundary_tolerance"] = 0.25
joblib.dump(cfg, "appeal_quality_config.joblib")

# Verify
cfg2 = joblib.load("appeal_quality_config.joblib")
print("Updated boundary_tolerance:", cfg2["boundary_tolerance"])
print("All keys:", list(cfg2.keys()))

Writing update_config.py


In [19]:
!python3.10 update_config.py

Updated boundary_tolerance: 0.25
All keys: ['label_classes', 'boundary_tolerance', 'lsa_components', 'scaler_feature_names']


In [20]:
from google.colab import files
for name in [
    "appeal_quality_vectorizer.joblib",
    "appeal_quality_vectorizer_si.joblib",
    "appeal_quality_vectorizer_ta.joblib",
    "appeal_quality_svd_en.joblib",
    "appeal_quality_svd_si.joblib",
    "appeal_quality_svd_ta.joblib",
    "appeal_quality_scaler.joblib",
    "appeal_quality_classifier.joblib",
    "appeal_quality_label_encoder.joblib",
    "appeal_quality_regressor.joblib",
    "appeal_quality_isotonic.joblib",
    "appeal_quality_config.joblib",
]:
    files.download(name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>